# Pulpermill backend (Kaggle launcher)
Settings: **Accelerator = GPU T4 x2**, **Internet = on**. Add two secrets (Add-ons, Secrets): `NGROK_AUTHTOKEN` and `PULPERMILL_TOKEN` (any long random string; type the same one into the page). Optionally add `HF_TOKEN` (a Hugging Face read token): the model then downloads signed in, which is faster.
No logic lives here: everything is in `pulpermill/backend/launch.py`. Run the cells in order.

## 1. Install

In [ ]:
import os, shutil, subprocess, sys
BRANCH = "main"
REPO = "https://github.com/LacChe/pulpermill"
DIR = "/kaggle/working/pulpermill"
if "pulpermill" in sys.modules:
    raise RuntimeError("The code was already imported in this session. Restart the session (Run > Restart) and run this cell first, so the fresh clone is the one that runs.")
os.chdir("/kaggle/working")
shutil.rmtree(DIR, ignore_errors=True)         # a fresh clone every session: an old copy would keep running the old code
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, DIR], check=True)
os.chdir(DIR)
sys.path.insert(0, DIR)
print("running commit", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "pulpermill/backend/requirements.txt", "pyngrok", "huggingface_hub"], check=True)


## 2. llama-cpp-python with CUDA
Loads the CUDA 12 runtime libraries (Kaggle's image may carry only CUDA 13), then uses the llama-cpp-python that is installed if it loads, and otherwise installs a ready-made CUDA 12 build. Safe to run again, and it must be run again after a restart. The first run takes a few minutes.

In [ ]:
import os, sys, glob, ctypes, site, importlib, importlib.util, subprocess

def pip(*args):
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args]).returncode

# 1. The CUDA 12 runtime libraries the llama-cpp-python wheels link against. The image may carry only CUDA 13, so they are found on the
#    system or taken from pip, then loaded into this process first (RTLD_GLOBAL) so the loader finds them by name when libllama.so loads.
#    Changing LD_LIBRARY_PATH now would not help: it is read when the process starts. After a restart, run this cell again.
NEEDED = ["libcudart.so.12", "libcublasLt.so.12", "libcublas.so.12"]

def search_dirs():
    roots = ["/usr/local/cuda*/lib64", "/usr/local/cuda*/targets/*/lib", "/usr/local/nvidia/lib64",
             "/usr/lib/x86_64-linux-gnu", "/opt/conda/lib"]
    for sp in site.getsitepackages() + [site.getusersitepackages()]:
        roots.append(sp + "/nvidia/*/lib")
    return [d for r in roots for d in glob.glob(r)]

def preload(name):
    try:
        ctypes.CDLL(name, mode=ctypes.RTLD_GLOBAL)
        return f"{name}: on the system path"
    except OSError:
        pass
    for d in search_dirs():
        for path in sorted(glob.glob(os.path.join(d, name + "*"))):
            try:
                ctypes.CDLL(path, mode=ctypes.RTLD_GLOBAL)
                return f"{name}: {path}"
            except OSError:
                continue
    return None

found = {n: preload(n) for n in NEEDED}
if not all(found.values()):
    print("CUDA 12 libraries not found:", [n for n, v in found.items() if not v], "- installing them from pip")
    if pip("nvidia-cuda-runtime-cu12", "nvidia-cublas-cu12"):
        raise RuntimeError("The CUDA 12 runtime packages could not be installed (pip's output is above).")
    importlib.invalidate_caches()
    found = {n: preload(n) for n in NEEDED}
for n, v in found.items():
    print(v or f"{n}: NOT FOUND")
if not all(found.values()):
    raise RuntimeError("A CUDA 12 library is still missing. Check that the accelerator is set to GPU T4 x2.")

# 2. The package. Kaggle's image may ship a llama-cpp-python that does not load, so it is replaced when the import fails.
#    The replacement is a ready-made CUDA 12 wheel from that index only, so pip cannot pick a source package and compile a CPU build.
#    --no-deps keeps pip from touching the image's own packages, so the few dependencies the wheel needs are installed by name.
def try_import():
    try:
        import llama_cpp
        return llama_cpp
    except Exception as e:
        print("llama-cpp-python does not load:", type(e).__name__, e)
        for name in [m for m in sys.modules if m == "llama_cpp" or m.startswith("llama_cpp.")]:
            del sys.modules[name]            # drop a half-loaded module so a fresh install is the one imported
        return None

llama_cpp = try_import()
if llama_cpp is None:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "llama-cpp-python"])
    for cuda in ("cu121", "cu124"):
        if pip("llama-cpp-python", "--no-deps", "--only-binary=:all:", "--index-url",
               f"https://abetlen.github.io/llama-cpp-python/whl/{cuda}") == 0:
            break
    else:
        raise RuntimeError("No ready-made llama-cpp-python wheel could be installed (pip's output is above).")
    if pip("diskcache", "jinja2", "typing-extensions", "numpy"):
        raise RuntimeError("The dependencies of llama-cpp-python could not be installed (pip's output is above).")
    importlib.invalidate_caches()
    llama_cpp = try_import()
    if llama_cpp is None:
        raise RuntimeError("llama-cpp-python was installed but still does not load (the message is above).")

# 3. Check that it can use the GPU.
if not llama_cpp.llama_supports_gpu_offload():
    raise RuntimeError("llama-cpp-python loaded but cannot use the GPU. The model would run on the CPU, far too slowly.")
print("llama-cpp-python", llama_cpp.__version__, "GPU offload: supported")


## 3. Load the model, start the server, open the tunnel
This cell keeps running; the address is printed when it is ready.


**Model file:** if a dataset attached to the notebook (Add Input) holds `mlabonne_gemma-3-27b-it-abliterated-Q6_K.gguf`, it is used as it is and nothing is downloaded. Otherwise it downloads to `/kaggle/temp/models`, which has far more room than `/kaggle/working`.


In [ ]:
from kaggle_secrets import UserSecretsClient
from pulpermill.backend.launch import launch
secrets = UserSecretsClient()
def optional(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        return None            # the secret is not set: carry on without it
# prompter: the wording module, a folder under pulpermill/prompter/ ("stub" answers with placeholders; "mill" is the real one under test)
# debug=True writes every refused reply and its problem line to /kaggle/working/pulpermill-debug.jsonl: it holds story text, so do not share or commit it
launch(token=secrets.get_secret("PULPERMILL_TOKEN"), ngrok_authtoken=secrets.get_secret("NGROK_AUTHTOKEN"), hf_token=optional("HF_TOKEN"), prompter="mill", debug=True, trace=True)

## Measuring (run once, in a separate session or after stopping cell 3)
Fills in `docs/backend-notes.md`: tokenizer parity, speed, cancellation, memory. Copy the printed Markdown back.

In [ ]:
from dataclasses import replace
from pulpermill.backend.config import Config
from pulpermill.backend.engine import download
from pulpermill.backend.engine.llama import load
from tools.measure_engine import measure_all, memory_by_ctx
model = Config().model
path = download.ensure_model(model)
engine = load(model, path)
measure_all(engine)

## Trials on the real model (for the prompts work)
Runs a chapter loop (the old `tools/run_trials.py` is at the git tag `pre-prompt-design`) in this session's engine, with no server. Needs the engine loaded as in the measuring cell. Each call's sampling and token/second record is kept in `rows`.

In [ ]:
from pulpermill.backend.complete import make_complete
from pulpermill.backend.config import Config

rows = []
complete = make_complete(engine, max_tokens=1600, params=Config().prose, record=rows)
# request = <a generate request dict, e.g. from evals/trials>
# text = complete(prompt)  # prompt design supplies the trial loop


In [ ]:
# The run log: one line per call (settings first). Paste the lines you want checked.
import json
for line in open("/kaggle/working/pulpermill-runs.jsonl").read().splitlines()[-20:]:
    r = json.loads(line)
    print(r["record"], r.get("call", ""), r.get("result", ""), r.get("seconds", ""), "retries", r.get("retries", ""), "squeeze", r.get("squeeze", ""))


In [ ]:
del engine
memory_by_ctx(model, path, [8192, 12288, 16384])